<a href="https://colab.research.google.com/github/bharathi-1022/DS_LAB/blob/main/EXP8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
'''
Experiment Title

Building Machine Learning Models using Spark MLlib (Logistic Regression and Decision Tree)

Aim

To build and evaluate machine learning classification models using Spark MLlib, specifically Logistic Regression and Decision Tree classifiers, on large-scale datasets in a distributed computing environment.

Description

With the growth of big data, traditional machine learning tools struggle to process large-scale datasets efficiently. Apache Spark provides a distributed computing framework that enables scalable and parallel data processing. Spark MLlib is the machine learning library of Apache Spark designed to build distributed ML models.

In this experiment, machine learning models are developed using Spark MLlib to perform classification tasks on large datasets.

1️⃣ Apache Spark MLlib

Apache Spark MLlib is a scalable machine learning library that supports distributed model training. It provides:

Classification algorithms
Regression models
Clustering techniques
Feature engineering tools
Model evaluation metrics

MLlib operates in a distributed manner, allowing models to be trained on large datasets efficiently.

2️⃣ Logistic Regression

Logistic Regression is a supervised classification algorithm used for binary classification problems.

Key Characteristics:
Probabilistic model
Uses Sigmoid function
Outputs probability between 0 and 1
Suitable for linearly separable data



In Spark MLlib, Logistic Regression supports distributed gradient descent optimization.

3️⃣ Decision Tree Classifier

Decision Tree is a non-linear supervised learning algorithm used for classification and regression.

Key Characteristics:
Tree-based structure
Splits data using impurity measures (Gini/Entropy)
Easy to interpret
Handles both numerical and categorical data

Decision Trees are suitable for complex decision boundaries but may overfit without proper tuning.

4️⃣ Steps Performed in the Experiment
Initialize Spark Session
Load large dataset
Perform feature transformation using VectorAssembler
Split dataset into training and testing sets
Train Logistic Regression model
Train Decision Tree model
Evaluate models using AUC and Accuracy
Compare model performance
5️⃣ Performance Evaluation

The models are evaluated using:

AUC (Area Under ROC Curve)
Measures the ability of the classifier to distinguish between classes.
Accuracy
Measures the proportion of correctly classified instances.
'''

# ==========================================================
# Build ML Models using Spark MLlib
# Logistic Regression and Decision Tree
# ==========================================================

!pip install pyspark
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# ==========================================================
# 1. Create Spark Session
# ==========================================================

spark = SparkSession.builder \
    .appName("Spark ML Models") \
    .getOrCreate()

print("Spark Session Started")

# ==========================================================
# 2. Load Dataset
# ==========================================================

# Replace with your dataset path
file_path = "/content/cleaned_data.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)

df.printSchema()
df.show(5)

# ==========================================================
# 3. Feature Engineering (VectorAssembler)
# ==========================================================

# Capture the name of the target column before the VectorAssembler transformation
original_target_column_name = df.columns[-1]

# Exclude target column from features
feature_columns = df.columns[:-1]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df = assembler.transform(df)

# Select only features and label, using the captured original target column name
data = df.select("features", df[original_target_column_name].alias("label"))

data.show(5)

# ==========================================================
# 4. Train-Test Split
# ==========================================================

train_data, test_data = data.randomSplit([0.7, 0.3], seed=42)

print("Training Data Count:", train_data.count())
print("Testing Data Count:", test_data.count())

# ==========================================================
# 5. Logistic Regression Model
# ==========================================================

lr = LogisticRegression(featuresCol="features", labelCol="label")

lr_model = lr.fit(train_data)

lr_predictions = lr_model.transform(test_data)

print("\n===== Logistic Regression Predictions =====")
lr_predictions.select("features", "label", "prediction").show(5)

# ==========================================================
# 6. Decision Tree Model
# ==========================================================

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")

dt_model = dt.fit(train_data)

dt_predictions = dt_model.transform(test_data)

print("\n===== Decision Tree Predictions =====")
dt_predictions.select("features", "label", "prediction").show(5)

# ==========================================================
# 7. Model Evaluation
# ==========================================================

# AUC Evaluation
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

lr_auc = evaluator_auc.evaluate(lr_predictions)
dt_auc = evaluator_auc.evaluate(dt_predictions)

# Accuracy Evaluation
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

lr_acc = evaluator_acc.evaluate(lr_predictions)
dt_acc = evaluator_acc.evaluate(dt_predictions)

print("\n=========== MODEL PERFORMANCE ===========")
print("Logistic Regression AUC:", lr_auc)
print("Logistic Regression Accuracy:", lr_acc)

print("\nDecision Tree AUC:", dt_auc)
print("Decision Tree Accuracy:", dt_acc)

# ==========================================================
# 8. Stop Spark Session
# ==========================================================

spark.stop()
print("Spark Session Stopped")

Spark Session Started
root
 |-- age: double (nullable = true)
 |-- sex: double (nullable = true)
 |-- cp: double (nullable = true)
 |-- trestbps: integer (nullable = true)
 |-- chol: integer (nullable = true)
 |-- fbs: integer (nullable = true)
 |-- restecg: integer (nullable = true)
 |-- thalachh: integer (nullable = true)
 |-- exang: integer (nullable = true)
 |-- oldpeak: double (nullable = true)
 |-- slope: integer (nullable = true)
 |-- ca: integer (nullable = true)
 |-- thal: integer (nullable = true)
 |-- target: double (nullable = true)

+----+---+---+--------+----+---+-------+--------+-----+-------+-----+---+----+------+
| age|sex| cp|trestbps|chol|fbs|restecg|thalachh|exang|oldpeak|slope| ca|thal|target|
+----+---+---+--------+----+---+-------+--------+-----+-------+-----+---+----+------+
|63.0|1.0|3.0|      31|  78|  1|      0|      47|    0|    2.3|    0|  0|   1|   1.0|
|37.0|1.0|2.0|      22|  95|  0|      1|      83|    0|    3.5|    0|  0|   2|   1.0|
|41.0|0.0|1.0|    